## Imports

In [ ]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib

from slack_sdk import WebClient

from scipy import stats

from tqdm.notebook import tqdm

from collections import Counter

from astropy.coordinates import SkyCoord
from astropy.table import Table, join, vstack, hstack
from astropy import units as u
from astropy import table

from RACSQuery import *
from RACSUtils import *

## Obtaining Info on Full RACSMid1 Fields

In [ ]:
# Create a new dataframe with the RACSLow field names and their corresponding SBID and UTC Scan Start Time
df_list = pd.DataFrame(np.load('RACSMid1_FullList_v2.npy', allow_pickle=True), columns=['Field Name', 'SBID', 'CAL_SBID', 'UTC Scan Start Time'])

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Set the reading directory path for the RACSMid1 beam information (epoch 1)
directory_path = 'epoch_1/'

# Create the saving directory if it doesn't exist
directory_main = 'D:\ASKAP Astrometry Storage'

directory = os.path.join(directory_main, 'RACSMid_Queries')
os.makedirs(directory, exist_ok=True)

# Create an empty list to store RACSLow scan information
df_racs = []

for i in range(len(df_list)):
    filename = f'beam_inf_{df_list["SBID"].values[i]}-{df_list["Field Name"].values[i]}.csv'
    filepath = os.path.join(directory_path, filename)
    
    # Read the RACS beam information to get beam number and beam center
    df = pd.read_csv(filepath)
    
    df['FIELD_NAME'] = filename.split('.')[0][-12:]
    df['SBID'] = int(filename.split('.')[0].split('beam_inf_')[1][:-13])
    df['CAL_SBID'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['CAL_SBID'].values[0]
    df['UTC_SCAN_START'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['UTC Scan Start Time'].values[0]
    
    # Append the dataframe to the list
    df_racs.append(df)

### Parameters for Modelling

In [ ]:
# Search radius in degrees
radius = 1.0
# Crossmatch threshold in arcsec
threshold0 = 12.0*u.arcsec
threshold1 = 5.0*u.arcsec
threshold2 = 2.0*u.arcsec
# Floor value in arcsec
floor = 0.4

## RACSMid1 Random Tests

In [ ]:
# Final RACSMid1 catalogue
racsmid1_catalogue = pd.read_csv(f'RACSMid1_Catalogue_AllGaussians_FullField_0.5degBeam_Corrected_Final.csv')

In [ ]:
# Crossmatched catalogues
racs_rfc_final = pd.read_csv(f'{directory}/RACSMid1_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')
racs_rfc_fil_final = pd.read_csv(f'{directory}/RACSMid1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')

In [ ]:
# Using RFC catalogue and RACSMid1 corrected source lists
directory_rfc = os.path.join(directory_main, 'RFC_Queries')

rfc_final = Table.read(os.path.join(directory_rfc, "rfc_combined.fits"), format='fits')
rfc_coord = SkyCoord(rfc_final['RAJ2000'], rfc_final['DEJ2000'], unit=u.degree)

In [ ]:
def Clean_RACSMid(df_racs, racs_raw):
# ## Cleaning the RACS catalogue as per our requirements:
#     remove sources with neighbouring sources < 30 arcsec
#     remove non-point sources (int/peak < 1.5)
#     snr > 6

    racs_final = []

    for i in range(len(df_racs)):
        racs_coord = SkyCoord(racs_raw[i]['ra_fin'], racs_raw[i]['dec_fin'], unit=u.degree)
        
        try:
            _, sep, _ = racs_coord.match_to_catalog_sky(racs_coord, nthneighbor=2)
            compactness = racs_raw[i]['int_flux'] / racs_raw[i]['peak_flux']
            snr = racs_raw[i]['peak_flux'] / racs_raw[i]['err_peak_flux']
            
            racs = racs_raw[i][(sep > 30*u.arcsec) & (compactness < 1.5) & (snr > 6)]
        
        except:
            racs = racs_raw[i].copy()
        
        racs_final.append(racs)
        
    return racs_final 

In [ ]:
# First random search. Only use when new random scans and beams are required
directory_racsmid = os.path.join(directory, 'RACSMid_Final_S2')

crop_radius = 0.4*u.degree

rfc_crop_list = []
racs_crop_list = []
racs_crop_fil_list = []
field_name_list = []
beam_num_list = []

for _ in tqdm(range(len(df_racs)), desc='RACSMid1 vs RFC', unit='plot'):
    k = np.random.randint(77, len(df_racs))
    
    field_name = df_racs[k].iloc[0]['FIELD_NAME']
    utc_time = str(df_racs[k].iloc[0]['UTC_SCAN_START'])[0:10]
    sbid_val = df_racs[k].iloc[0]['SBID']
    save_racs_query = os.path.join(directory_racsmid, f'{utc_time}_RACSMid_Queries_SB{sbid_val}_{field_name[-7:]}_rad{radius}')

    racs_final = [Table(np.load(os.path.join(save_racs_query, f'Beam_{i}.npy'))) for i in range(len(os.listdir(save_racs_query))) 
                if os.path.isfile(os.path.join(save_racs_query, f'Beam_{i}.npy'))]
    racs_fil_final = Clean_RACSMid(df_racs[k], racs_final)

    print(f"Running Scan {k} (Field: {field_name})")

    for _ in range(len(df_racs[k])):
        i = np.random.randint(0, len(df_racs[k]))
        print(f"Running Beam {i} of Scan {k} (Field: {field_name})")
        
        # crop a circle of crop_radius from the centre of beam
        beam_coord = SkyCoord(df_racs[k]['RA_DEG'].values[i], df_racs[k]['DEC_DEG'].values[i], unit=u.degree)
        
        sep_rfc = beam_coord.separation(rfc_coord)
        rfc_crop = rfc_final[sep_rfc < crop_radius]
        
        zz = False
        for j in range(len(rfc_crop)):
            if rfc_crop['JNAME'][j] in racs_rfc_final['JNAME'].values:
                print(f"Beam {i} of Scan {k} (Field: {field_name}) has a crossmatched source with RFC catalogue.")
                rfc_crop_new = rfc_crop[rfc_crop['JNAME'] == rfc_crop['JNAME'][j]]
                zz = True

        if zz:
            rfc_crop_coord = SkyCoord(rfc_crop_new['RAJ2000'], rfc_crop_new['DEJ2000'], unit=u.degree)
            
            racs_coord = SkyCoord(racs_final[i]['ra_fin'], racs_final[i]['dec_fin'], unit=u.degree)
            sep_racs = beam_coord.separation(racs_coord)
            racs_crop = racs_final[i][sep_racs < crop_radius]
            racs_crop_coord = SkyCoord(racs_crop['ra_fin'], racs_crop['dec_fin'], unit=u.degree)
            
            racs_fil_coord = SkyCoord(racs_fil_final[i]['ra_fin'], racs_fil_final[i]['dec_fin'], unit=u.degree)
            sep_racs_fil = beam_coord.separation(racs_fil_coord)
            racs_crop_fil = racs_fil_final[i][sep_racs_fil < crop_radius]
            racs_crop_fil_coord = SkyCoord(racs_crop_fil['ra_fin'], racs_crop_fil['dec_fin'], unit=u.degree)
            
            break
    
    if zz:    
        rfc_crop_list.append(rfc_crop_new)
        racs_crop_list.append(racs_crop)
        racs_crop_fil_list.append(racs_crop_fil)
        field_name_list.append(field_name)
        beam_num_list.append(i)
    
    
    if len(rfc_crop_list) == 100:
        break

In [ ]:
beam_num_list = []

In [ ]:
field_num_list = [373, 726, 742, 925, 1218, 196, 233, 657, 138, 1375]
beam_num_list = [32, 26, 28, 30, 1, 19, 31, 26, 32, 24]

In [ ]:
# Repeat search. Use when the same scans and beams are required for further analysis or plotting
directory_racsmid = os.path.join(directory, 'RACSMid_Final_S2')

crop_radius = 0.4*u.degree

rfc_crop_list = []
racs_crop_list = []
racs_crop_fil_list = []
field_name_list = []
# beam_num_list = []

for kzz in tqdm(range(10), desc='RACSMid1 vs RFC', unit='plot'):
    # k = np.random.randint(77, len(df_racs))
    k = field_num_list[kzz]
    i = beam_num_list[kzz]
    
    field_name = df_racs[k].iloc[0]['FIELD_NAME']
    utc_time = str(df_racs[k].iloc[0]['UTC_SCAN_START'])[0:10]
    sbid_val = df_racs[k].iloc[0]['SBID']
    save_racs_query = os.path.join(directory_racsmid, f'{utc_time}_RACSMid_Queries_SB{sbid_val}_{field_name[-7:]}_rad{radius}')

    racs_final = [Table(np.load(os.path.join(save_racs_query, f'Beam_{i}.npy'))) for i in range(len(os.listdir(save_racs_query))) 
                if os.path.isfile(os.path.join(save_racs_query, f'Beam_{i}.npy'))]
    racs_fil_final = Clean_RACSMid(df_racs[k], racs_final)

    print(f"Running Scan {k} (Field: {field_name})")

    for _ in range(1):
        # i = np.random.randint(0, len(df_racs[k]))
        print(f"Running Beam {i} of Scan {k} (Field: {field_name})")
        
        # crop a circle of 1 deg radius from the centre of beam
        beam_coord = SkyCoord(df_racs[k]['RA_DEG'].values[i], df_racs[k]['DEC_DEG'].values[i], unit=u.degree)
        
        sep_rfc = beam_coord.separation(rfc_coord)
        rfc_crop = rfc_final[sep_rfc < crop_radius]
        
        zz = False
        for j in range(len(rfc_crop)):
            if rfc_crop['JNAME'][j] in racs_rfc_final['JNAME'].values:
                print(f"Beam {i} of Scan {k} (Field: {field_name}) has a crossmatched source with RFC catalogue.")
                rfc_crop_new = rfc_crop[rfc_crop['JNAME'] == rfc_crop['JNAME'][j]]
                zz = True

        if zz:
            rfc_crop_coord = SkyCoord(rfc_crop_new['RAJ2000'], rfc_crop_new['DEJ2000'], unit=u.degree)
            
            racs_coord = SkyCoord(racs_final[i]['ra_fin'], racs_final[i]['dec_fin'], unit=u.degree)
            sep_racs = beam_coord.separation(racs_coord)
            racs_crop = racs_final[i][sep_racs < crop_radius]
            racs_crop_coord = SkyCoord(racs_crop['ra_fin'], racs_crop['dec_fin'], unit=u.degree)
            
            racs_fil_coord = SkyCoord(racs_fil_final[i]['ra_fin'], racs_fil_final[i]['dec_fin'], unit=u.degree)
            sep_racs_fil = beam_coord.separation(racs_fil_coord)
            racs_crop_fil = racs_fil_final[i][sep_racs_fil < crop_radius]
            racs_crop_fil_coord = SkyCoord(racs_crop_fil['ra_fin'], racs_crop_fil['dec_fin'], unit=u.degree)
            
            break
    
    if zz:    
        rfc_crop_list.append(rfc_crop_new)
        racs_crop_list.append(racs_crop)
        racs_crop_fil_list.append(racs_crop_fil)
        field_name_list.append(field_name)
        # beam_num_list.append(i)
    
    
    if len(rfc_crop_list) == 10:
        break

In [ ]:
# make a 2x5 plot and plot racs_crop_list, racs_crop_fil_list, and rfc_crop_list.
plt.figure(figsize=(50, 60))
for i, (rfc_crop, racs_crop, racs_crop_fil, field_name, beam_num) in enumerate(zip(rfc_crop_list, racs_crop_list, racs_crop_fil_list, field_name_list, beam_num_list)):
    plt.subplot(10, 10, i+1)
    rfc_crop_coord = SkyCoord(rfc_crop['RAJ2000'], rfc_crop['DEJ2000'], unit=u.degree)
    racs_crop_coord = SkyCoord(racs_crop['ra_fin'], racs_crop['dec_fin'], unit=u.degree)
    racs_crop_fil_coord = SkyCoord(racs_crop_fil['ra_fin'], racs_crop_fil['dec_fin'], unit=u.degree)

    plt.scatter(racs_crop_coord.ra, racs_crop_coord.dec, s=15, marker='o', color='green')
    plt.scatter(racs_crop_fil_coord.ra, racs_crop_fil_coord.dec, s=10, marker='+', color='blue')
    plt.scatter(rfc_crop_coord.ra, rfc_crop_coord.dec, s=10, marker='x', color='red')
    plt.xlabel('Right Ascension (deg)')
    plt.ylabel('Declination (deg)')
    plt.legend(['RACSMid1 Sources', 'Filtered RACSMid1 Sources', 'RFC Sources: ' + rfc_crop['JNAME'][0]])
    plt.title(f'{len(racs_crop)} RACSMid1 Sources and {len(rfc_crop)} RFC Sources \nin {field_name} Beam {beam_num}')
plt.show()

In [ ]:
radius_plot = 30.0 * u.arcsec
delta_deg = radius_plot.to(u.deg).value

plt.figure(figsize=(50, 60))

for i, (rfc_crop, racs_crop, racs_crop_fil, field_name, beam_num) in enumerate(
    zip(rfc_crop_list, racs_crop_list, racs_crop_fil_list, field_name_list, beam_num_list)):

    plt.subplot(10, 10, i + 1)

    rfc_crop_coord = SkyCoord(rfc_crop['RAJ2000'], rfc_crop['DEJ2000'], unit=u.degree)
    rfc_center = rfc_crop_coord[0]

    racs_crop_coord = SkyCoord(racs_crop['ra_fin'], racs_crop['dec_fin'], unit=u.degree)
    racsmid1_cat_coord = SkyCoord(racsmid1_catalogue['ra_fin'], racsmid1_catalogue['dec_fin'], unit=u.degree)
    racs_crop_fil_coord = SkyCoord(racs_crop_fil['ra_fin'], racs_crop_fil['dec_fin'], unit=u.degree)

    racs_in = racs_crop[rfc_center.separation(racs_crop_coord) <= radius_plot]
    racs_cat_in = racsmid1_catalogue[rfc_center.separation(racsmid1_cat_coord) <= radius_plot]
    racs_fil_in = racs_crop_fil[rfc_center.separation(racs_crop_fil_coord) <= radius_plot]
    rfc_in = rfc_crop[rfc_center.separation(rfc_crop_coord) <= radius_plot]

    plt.scatter(racs_in['ra_fin'], racs_in['dec_fin'], s=80, marker='o', color='green')
    plt.scatter(racs_cat_in['ra_fin'], racs_cat_in['dec_fin'], s=50, marker=',', color='cyan')
    plt.scatter(racs_fil_in['ra_fin'], racs_fil_in['dec_fin'], s=35, marker='+', color='blue')
    plt.scatter(rfc_in['RAJ2000'], rfc_in['DEJ2000'], s=20, marker='x', color='red')

    circle = plt.Circle((rfc_center.ra.deg, rfc_center.dec.deg),
            delta_deg, fill=False, edgecolor='black', linewidth=1)
    plt.gca().add_patch(circle)

    plt.xlabel('Right Ascension (deg)')
    plt.ylabel('Declination (deg)')
    plt.legend([f'RACSMid1 Source: {racs_in["uuid"][0]}', f'Catalogue RACSMid1 Source: {racs_cat_in["uuid"].iloc[0]}', f'Filtered RACSMid1 Source: {racs_fil_in["uuid"][0] if len(racs_fil_in) > 0 else "null"}', f'RFC Source: {rfc_crop["JNAME"][0]}'], loc='upper right', fontsize=6)
    plt.title(f'{field_name} Beam {beam_num}: {len(racs_in)} RACSMid1, \n{len(racs_cat_in)} catalogue, {len(racs_fil_in)} filtered, {len(rfc_in)} RFC')
    plt.xlim(rfc_center.ra.deg - delta_deg, rfc_center.ra.deg + delta_deg)
    plt.ylim(rfc_center.dec.deg - delta_deg, rfc_center.dec.deg + delta_deg)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.ticklabel_format(useOffset=False)

plt.tight_layout()
plt.show()

In [ ]:
# list of all JNAMEs in rfc_crop_list
jname_list = []
for rfc_crop in rfc_crop_list:
    for jname in rfc_crop['JNAME']:
        jname_list.append(jname)

jname_list

In [ ]:
# table of all rows in racs_rfc_final with JNAME in jname_list
racs_rfc_final_crop = racs_rfc_final[racs_rfc_final['JNAME'].isin(jname_list)]
# remove rows with duplicate JNAMEs in racs_rfc_final_crop, keeping the first occurrence and print the number of duplicate JNAMEs removed
num_duplicates = racs_rfc_final_crop.duplicated(subset='JNAME').sum()
print(f'Number of duplicate JNAMEs removed: {num_duplicates}')
racs_rfc_final_crop = racs_rfc_final_crop.drop_duplicates(subset='JNAME', keep='first')
# arrange the rows in racs_rfc_final_crop in the same order as jname_list
racs_rfc_final_crop = racs_rfc_final_crop.set_index('JNAME').reindex(jname_list).reset_index()

racs_rfc_final_crop

In [ ]:
racs_rfc_final_crop['compactness'] = racs_rfc_final_crop['int_flux'] / racs_rfc_final_crop['peak_flux']
racs_rfc_final_crop['snr'] = racs_rfc_final_crop['peak_flux'] / racs_rfc_final_crop['err_peak_flux']

# if compactness > 1.5 or snr < 6, then add a new column 'flag' with value 'non-point source' or 'low snr' respectively, else 'point source'
def flag_source(row):
    if row['compactness'] > 1.5 and row['snr'] < 6:
        return 'non-point source and low snr'
    elif row['compactness'] > 1.5:
        return 'non-point source'
    elif row['snr'] < 6:
        return 'low snr'
    else:
        return 'point source'

racs_rfc_final_crop['flag'] = racs_rfc_final_crop.apply(flag_source, axis=1)

racs_rfc_final_crop

In [ ]:
racs_rfc_final_crop['sep']*u.degree.to(u.arcsec)

In [ ]:
plt.figure(figsize=(30, 14))

for i in range(len(racs_rfc_final_crop)):
    plt.subplot(2, 5, i + 1)

    rfc_crop_coord_2 = SkyCoord(racs_rfc_final_crop['RAJ2000'][i], racs_rfc_final_crop['DEJ2000'][i], unit=u.degree)
    racs_crop_coord_2 = SkyCoord(racs_rfc_final_crop['ra_fin'][i], racs_rfc_final_crop['dec_fin'][i], unit=u.degree)

    plt.scatter(racs_crop_coord_2.ra, racs_crop_coord_2.dec, s=60, marker='o', color='green')
    plt.scatter(rfc_crop_coord_2.ra, rfc_crop_coord_2.dec, s=30, marker='x', color='red')

    circle_2 = plt.Circle((rfc_crop_coord_2.ra.deg, rfc_crop_coord_2.dec.deg),
            delta_deg, fill=False, edgecolor='black', linewidth=1)
    plt.gca().add_patch(circle_2)

    plt.xlabel('Right Ascension (deg)')
    plt.ylabel('Declination (deg)')
    plt.legend([f'RACSMid1 Sources {racs_rfc_final_crop["uuid"][i]}', f'RFC Source {racs_rfc_final_crop["JNAME"][i]}'], loc='upper right')
    plt.title(f'Crossmatched RACSMid1 and RFC Sources: {racs_rfc_final_crop["sep"][i]*u.degree.to(u.arcsec):.2f} arcsec')
    plt.xlim(rfc_crop_coord_2.ra.deg - delta_deg, rfc_crop_coord_2.ra.deg + delta_deg)
    plt.ylim(rfc_crop_coord_2.dec.deg - delta_deg, rfc_crop_coord_2.dec.deg + delta_deg)
    plt.gca().set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

In [ ]:
racs_rfc_fil_final_crop = racs_rfc_fil_final[racs_rfc_fil_final['JNAME'].isin(jname_list)]

num_duplicates_fil = racs_rfc_fil_final_crop.duplicated(subset='JNAME').sum()
print(f'Number of duplicate JNAMEs removed: {num_duplicates_fil}')
racs_rfc_fil_final_crop = racs_rfc_fil_final_crop.drop_duplicates(subset='JNAME', keep='first')

racs_rfc_fil_final_crop = racs_rfc_fil_final_crop.set_index('JNAME').reindex(jname_list).reset_index()

racs_rfc_fil_final_crop

## Obtaining Info on Full RACSHigh1 Fields

In [ ]:
# Create a new dataframe with the RACSLow field names and their corresponding SBID and UTC Scan Start Time
df_list = pd.DataFrame(np.load('RACSHigh1_FullList_v1.npy', allow_pickle=True), columns=['Field Name', 'SBID', 'CAL_SBID', 'UTC Scan Start Time'])

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Set the reading directory path for the RACSMid1 beam information (epoch 1)
directory_path = 'epoch_5/'

# Create the saving directory if it doesn't exist
directory_main = 'D:\ASKAP Astrometry Storage'

directory = os.path.join(directory_main, 'RACSHigh_Queries')
os.makedirs(directory, exist_ok=True)

# Create an empty list to store RACSLow scan information
df_racs = []

for i in range(len(df_list)):
    filename = f'beam_inf_{df_list["SBID"].values[i]}-{df_list["Field Name"].values[i]}.csv'
    filepath = os.path.join(directory_path, filename)
    
    # Read the RACS beam information to get beam number and beam center
    df = pd.read_csv(filepath)
    
    df['FIELD_NAME'] = filename.split('.')[0][-12:]
    df['SBID'] = int(filename.split('.')[0].split('beam_inf_')[1][:-13])
    df['CAL_SBID'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['CAL_SBID'].values[0]
    df['UTC_SCAN_START'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['UTC Scan Start Time'].values[0]
    
    # Append the dataframe to the list
    df_racs.append(df)

## RACSHigh1 Random Tests

In [ ]:
# Final RACSHigh1 catalogue
racshigh1_catalogue = pd.read_csv(f'RACSHigh1_Catalogue_AllGaussians_FullField_0.5degBeam_Corrected_Final.csv')

In [ ]:
# Crossmatched catalogues
racs_rfc_final = pd.read_csv(f'{directory}/RACSHigh1_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')
racs_rfc_fil_final = pd.read_csv(f'{directory}/RACSHigh1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')

In [ ]:
def Clean_RACSHigh(df_racs, racs_raw):
# ## Cleaning the RACS catalogue as per our requirements:
#     remove sources with neighbouring sources < 30 arcsec
#     remove non-point sources (int/peak < 1.5)
#     snr > 6

    racs_final = []

    for i in range(len(df_racs)):
        racs_coord = SkyCoord(racs_raw[i]['col_ra_deg_fin'], racs_raw[i]['col_dec_deg_fin'], unit=u.degree)
        
        try:
            _, sep, _ = racs_coord.match_to_catalog_sky(racs_coord, nthneighbor=2)
            compactness = racs_raw[i]['col_flux_int'] / racs_raw[i]['col_flux_peak']
            snr = racs_raw[i]['col_flux_peak'] / racs_raw[i]['col_flux_peak_err']
            
            racs = racs_raw[i][(sep > 30*u.arcsec) & (compactness < 1.5) & (snr > 6)]
        
        except:
            racs = racs_raw[i].copy()
        
        racs_final.append(racs)
        
    return racs_final 

In [ ]:
# field_name_list = ['RACS_2300+28', 'RACS_0752+09', 'RACS_1316-14', 'RACS_1125-23', 'RACS_1645-32', 'RACS_1319+23', 'RACS_0412+28', 'RACS_2255+00', 'RACS_2236-28', 'RACS_0514+46']
# field_num_list = [531, 346, 1460, 682, 1208, 362, 173, 1086, 1003, 51]
# beam_num_list = [32, 26, 28, 30, 1, 19, 31, 26, 32, 24]

field_name_list = ['RACS_2141-69', 'RACS_1044+09', 'RACS_1252-78', 'RACS_0306+32', 'RACS_1146-41', 'RACS_0718+18', 'RACS_1812-28', 'RACS_0500+09', 'RACS_1356-64', 'RACS_2025-04', 'RACS_1257+23', 'RACS_0421-37', 'RACS_0830-64', 'RACS_1126-18', 'RACS_1039+23', 'RACS_0243-14', 'RACS_0147-04', 'RACS_0334-04', 'RACS_0248+18', 'RACS_1545+00', 'RACS_2155-41', 'RACS_0141-18', 'RACS_2042-55', 'RACS_2012-28', 'RACS_0813-60', 'RACS_0641-41', 'RACS_0526-73',
                'RACS_0151+32', 'RACS_2223-41', 'RACS_0118-37', 'RACS_0618-55', 'RACS_1315-04', 'RACS_0724-28', 'RACS_1839+23', 'RACS_1754+23', 'RACS_1610-14', 'RACS_0444+46', 'RACS_1920+00', 'RACS_0237-37', 'RACS_2108-09', 'RACS_0605-09', 'RACS_0543-51', 'RACS_0700+28', 'RACS_1144+46', 'RACS_2233+32', 'RACS_1412+28', 'RACS_2138+14', 'RACS_0835-09', 'RACS_1309-64', 'RACS_0306+32', 'RACS_1949-14', 'RACS_0331-32', 'RACS_1318-18', 'RACS_2133+18',
                'RACS_1149+04', 'RACS_1559-23', 'RACS_0000-73', 'RACS_1500-28','RACS_2025-04', 'RACS_2317+04', 'RACS_1502+09', 'RACS_0744-64', 'RACS_0216+32', 'RACS_0334-09', 'RACS_2237-60', 'RACS_2026-18', 'RACS_0454-14', 'RACS_1144+46', 'RACS_1239-37', 'RACS_0814-09', 'RACS_2317+09', 'RACS_2208+32', 'RACS_1607+09', 'RACS_2234+09', 'RACS_0835-09', 'RACS_0832-14', 'RACS_1149-09', 'RACS_0848+18', 'RACS_0751-37', 'RACS_1336+00', 'RACS_0334+00',
                'RACS_2054-14', 'RACS_0749-14', 'RACS_0204+41', 'RACS_1033+32', 'RACS_0606-37', 'RACS_0859-41', 'RACS_1936-41', 'RACS_1044+04','RACS_0605+04', 'RACS_0724-28', 'RACS_0303-37', 'RACS_1122+32', 'RACS_1001-09', 'RACS_2234-09', 'RACS_1120+37', 'RACS_2346-41', 'RACS_0804+41','RACS_1054+37', 'RACS_1821+14']

beam_num_list = [0, 31, 13, 26, 35, 13, 15, 0, 9, 5, 32, 32, 26, 2, 27, 16, 15, 10, 7, 15, 28, 33, 6, 4, 14, 20, 28, 27, 34, 16, 32, 29, 21, 13, 5, 6, 4, 21, 13, 19, 33, 2, 18, 25, 23, 19, 17, 17, 9, 11, 5, 17, 1, 11, 33, 9, 13, 11, 13, 35, 9, 18, 21, 31, 22, 16, 19, 18, 19, 28, 9, 32, 11, 0, 30, 6, 19, 29, 24, 18, 4, 7, 26, 35, 14, 32, 22, 32, 29, 0, 27, 2, 5, 26, 20, 30, 18, 12, 23, 17]

In [ ]:
# Repeat search. Use when the same scans and beams are required for further analysis or plotting
directory_racshigh = os.path.join(directory, 'RACSHigh_Final_S2')

crop_radius = 0.4*u.degree

rfc_crop_list = []
racs_crop_list = []
racs_crop_fil_list = []
field_name_list_high = []
beam_num_list_high = []

# kzz = 0
for k in tqdm(range(len(df_racs)), desc='RACSHigh1 vs RFC', unit='plot'):
    # k = np.random.randint(77, len(df_racs))
    # k = field_num_list[kzz]
    # i = beam_num_list[kzz]
    
    field_name = df_racs[k].iloc[0]['FIELD_NAME']
    
    # dont run loop if field_name is not in field_name_list
    if field_name not in field_name_list:
        continue
    
    # get index of field_name in field_name_list
    index = field_name_list.index(field_name)
    i = beam_num_list[index]
    
    utc_time = str(df_racs[k].iloc[0]['UTC_SCAN_START'])[0:10]
    sbid_val = df_racs[k].iloc[0]['SBID']
    save_racs_query = os.path.join(directory_racshigh, f'{utc_time}_RACSHigh_Queries_SB{sbid_val}_{field_name[-7:]}_rad{radius}')

    racs_final = [Table(np.load(os.path.join(save_racs_query, f'Beam_{i}.npy'))) for i in range(len(os.listdir(save_racs_query))) 
                if os.path.isfile(os.path.join(save_racs_query, f'Beam_{i}.npy'))]
    racs_fil_final = Clean_RACSHigh(df_racs[k], racs_final)

    print(f"Running Scan {k} (Field: {field_name})")

    for _ in range(1):
        # i = np.random.randint(0, len(df_racs[k]))
        print(f"Running Beam {i} of Scan {k} (Field: {field_name})")
        
        # crop a circle of crop_radius from the centre of beam
        beam_coord = SkyCoord(df_racs[k]['RA_DEG'].values[i], df_racs[k]['DEC_DEG'].values[i], unit=u.degree)
        
        sep_rfc = beam_coord.separation(rfc_coord)
        rfc_crop = rfc_final[sep_rfc < crop_radius]
        
        zz = False
        for j in range(len(rfc_crop)):
            if rfc_crop['JNAME'][j] in racs_rfc_final['JNAME'].values:
                print(f"Beam {i} of Scan {k} (Field: {field_name}) has a crossmatched source with RFC catalogue.")
                rfc_crop_new = rfc_crop[rfc_crop['JNAME'] == rfc_crop['JNAME'][j]]
                zz = True

        if zz:
            rfc_crop_coord = SkyCoord(rfc_crop_new['RAJ2000'], rfc_crop_new['DEJ2000'], unit=u.degree)
            
            racs_coord = SkyCoord(racs_final[i]['col_ra_deg_fin'], racs_final[i]['col_dec_deg_fin'], unit=u.degree)
            sep_racs = beam_coord.separation(racs_coord)
            racs_crop = racs_final[i][sep_racs < crop_radius]
            racs_crop_coord = SkyCoord(racs_crop['col_ra_deg_fin'], racs_crop['col_dec_deg_fin'], unit=u.degree)
            
            racs_fil_coord = SkyCoord(racs_fil_final[i]['col_ra_deg_fin'], racs_fil_final[i]['col_dec_deg_fin'], unit=u.degree)
            sep_racs_fil = beam_coord.separation(racs_fil_coord)
            racs_crop_fil = racs_fil_final[i][sep_racs_fil < crop_radius]
            racs_crop_fil_coord = SkyCoord(racs_crop_fil['col_ra_deg_fin'], racs_crop_fil['col_dec_deg_fin'], unit=u.degree)
            
            break
    
    if zz:    
        rfc_crop_list.append(rfc_crop_new)
        racs_crop_list.append(racs_crop)
        racs_crop_fil_list.append(racs_crop_fil)
        field_name_list_high.append(field_name)
        beam_num_list_high.append(i)
    
    # kzz += 1
    
    if len(rfc_crop_list) == 100:
        break

In [ ]:
# zip(rfc_crop_list, racs_crop_list, racs_crop_fil_list, field_name_list_high, beam_num_list_high) and reorder in the order of field_name_list
zipped_lists = zip(rfc_crop_list, racs_crop_list, racs_crop_fil_list, field_name_list_high, beam_num_list_high)
sorted_zipped_lists = sorted(zipped_lists, key=lambda x: field_name_list.index(x[3]))
# rfc_crop_list, racs_crop_list, racs_crop_fil_list, field_name_list_high, beam_num_list_high = zip(*sorted_zipped_lists)

In [ ]:
# make a 2x5 plot and plot racs_crop_list, racs_crop_fil_list, and rfc_crop_list.
plt.figure(figsize=(50, 60))
for i, (rfc_crop, racs_crop, racs_crop_fil, field_name, beam_num) in enumerate(sorted_zipped_lists):
    plt.subplot(10, 10, i+1)
    rfc_crop_coord = SkyCoord(rfc_crop['RAJ2000'], rfc_crop['DEJ2000'], unit=u.degree)
    racs_crop_coord = SkyCoord(racs_crop['col_ra_deg_fin'], racs_crop['col_dec_deg_fin'], unit=u.degree)
    racs_crop_fil_coord = SkyCoord(racs_crop_fil['col_ra_deg_fin'], racs_crop_fil['col_dec_deg_fin'], unit=u.degree)

    plt.scatter(racs_crop_coord.ra, racs_crop_coord.dec, s=15, marker='o', color='green')
    plt.scatter(racs_crop_fil_coord.ra, racs_crop_fil_coord.dec, s=10, marker='+', color='blue')
    plt.scatter(rfc_crop_coord.ra, rfc_crop_coord.dec, s=10, marker='x', color='red')
    plt.xlabel('Right Ascension (deg)')
    plt.ylabel('Declination (deg)')
    plt.legend(['RACSHigh1 Sources', 'Filtered RACSHigh1 Sources', 'RFC Sources: ' + rfc_crop['JNAME'][0]])
    plt.title(f'{len(racs_crop)} RACSHigh1 Sources and {len(rfc_crop)} RFC Sources \nin {field_name} Beam {beam_num}')
plt.show()

In [ ]:
radius_plot = 30.0 * u.arcsec
delta_deg = radius_plot.to(u.deg).value

plt.figure(figsize=(50, 60))

for i, (rfc_crop, racs_crop, racs_crop_fil, field_name, beam_num) in enumerate(sorted_zipped_lists):

    plt.subplot(10, 10, i + 1)

    rfc_crop_coord = SkyCoord(rfc_crop['RAJ2000'], rfc_crop['DEJ2000'], unit=u.degree)
    rfc_center = rfc_crop_coord[0]

    racs_crop_coord = SkyCoord(racs_crop['col_ra_deg_fin'], racs_crop['col_dec_deg_fin'], unit=u.degree)
    racshigh1_cat_coord = SkyCoord(racshigh1_catalogue['col_ra_deg_fin'], racshigh1_catalogue['col_dec_deg_fin'], unit=u.degree)
    racs_crop_fil_coord = SkyCoord(racs_crop_fil['col_ra_deg_fin'], racs_crop_fil['col_dec_deg_fin'], unit=u.degree)

    racs_in = racs_crop[rfc_center.separation(racs_crop_coord) <= radius_plot]
    racs_cat_in = racshigh1_catalogue[rfc_center.separation(racshigh1_cat_coord) <= radius_plot]
    racs_fil_in = racs_crop_fil[rfc_center.separation(racs_crop_fil_coord) <= radius_plot]
    rfc_in = rfc_crop[rfc_center.separation(rfc_crop_coord) <= radius_plot]

    plt.scatter(racs_in['col_ra_deg_fin'], racs_in['col_dec_deg_fin'], s=80, marker='o', color='green')
    plt.scatter(racs_cat_in['col_ra_deg_fin'], racs_cat_in['col_dec_deg_fin'], s=50, marker=',', color='cyan')
    plt.scatter(racs_fil_in['col_ra_deg_fin'], racs_fil_in['col_dec_deg_fin'], s=35, marker='+', color='blue')
    plt.scatter(rfc_in['RAJ2000'], rfc_in['DEJ2000'], s=20, marker='x', color='red')

    circle = plt.Circle((rfc_center.ra.deg, rfc_center.dec.deg),
            delta_deg, fill=False, edgecolor='black', linewidth=1)
    plt.gca().add_patch(circle)

    plt.xlabel('Right Ascension (deg)')
    plt.ylabel('Declination (deg)')
    plt.legend([f'RACSHigh1 Source: {racs_in["col_component_id"][0]}', f'Catalogue RACSHigh1 Source: {racs_cat_in["col_component_id"].iloc[0]}', f'Filtered RACSHigh1 Source: {racs_fil_in["col_component_id"][0] if len(racs_fil_in) > 0 else "null"}', f'RFC Source: {rfc_crop["JNAME"][0]}'], loc='upper right', fontsize=6)
    plt.title(f'{field_name} Beam {beam_num}: {len(racs_in)} RACSHigh1, \n{len(racs_cat_in)} catalogue, {len(racs_fil_in)} filtered, {len(rfc_in)} RFC')
    plt.xlim(rfc_center.ra.deg - delta_deg, rfc_center.ra.deg + delta_deg)
    plt.ylim(rfc_center.dec.deg - delta_deg, rfc_center.dec.deg + delta_deg)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.ticklabel_format(useOffset=False)

plt.tight_layout()
plt.show()

In [ ]:
# list of all JNAMEs in rfc_crop_list
jname_list = []
for rfc_crop in rfc_crop_list:
    for jname in rfc_crop['JNAME']:
        jname_list.append(jname)

jname_list

In [ ]:
# table of all rows in racs_rfc_final with JNAME in jname_list
racs_rfc_final_crop = racs_rfc_final[racs_rfc_final['JNAME'].isin(jname_list)]
# remove rows with duplicate JNAMEs in racs_rfc_final_crop, keeping the first occurrence and print the number of duplicate JNAMEs removed
num_duplicates = racs_rfc_final_crop.duplicated(subset='JNAME').sum()
print(f'Number of duplicate JNAMEs removed: {num_duplicates}')
racs_rfc_final_crop = racs_rfc_final_crop.drop_duplicates(subset='JNAME', keep='first')
# arrange the rows in racs_rfc_final_crop in the same order as jname_list
racs_rfc_final_crop = racs_rfc_final_crop.set_index('JNAME').reindex(jname_list).reset_index()

racs_rfc_final_crop

In [ ]:
racs_rfc_final_crop['compactness'] = racs_rfc_final_crop['col_flux_int'] / racs_rfc_final_crop['col_flux_peak']
racs_rfc_final_crop['snr'] = racs_rfc_final_crop['col_flux_peak'] / racs_rfc_final_crop['col_flux_peak_err']

# if compactness > 1.5 or snr < 6, then add a new column 'flag' with value 'non-point source' or 'low snr' respectively, else 'point source'
def flag_source(row):
    if row['compactness'] > 1.5 and row['snr'] < 6:
        return 'non-point source and low snr'
    elif row['compactness'] > 1.5:
        return 'non-point source'
    elif row['snr'] < 6:
        return 'low snr'
    else:
        return 'point source'

racs_rfc_final_crop['flag'] = racs_rfc_final_crop.apply(flag_source, axis=1)

racs_rfc_final_crop

In [ ]:
racs_rfc_final_crop['sep']*u.degree.to(u.arcsec)

In [ ]:
racs_rfc_fil_final_crop = racs_rfc_fil_final[racs_rfc_fil_final['JNAME'].isin(jname_list)]

num_duplicates_fil = racs_rfc_fil_final_crop.duplicated(subset='JNAME').sum()
print(f'Number of duplicate JNAMEs removed: {num_duplicates_fil}')
racs_rfc_fil_final_crop = racs_rfc_fil_final_crop.drop_duplicates(subset='JNAME', keep='first')

racs_rfc_fil_final_crop = racs_rfc_fil_final_crop.set_index('JNAME').reindex(jname_list).reset_index()

racs_rfc_fil_final_crop

## Extract Text

In [ ]:
text_file = "beam_text_test.txt"

# Open the text file in read mode
with open(text_file, "r") as f:
    text = f.read()
    
    # remove all duplicate lines
    lines = text.split("\n")
    unique_lines = []
    duplicate_lines = []
    for line in lines:
        if line not in unique_lines:
            unique_lines.append(line)
        else:
            duplicate_lines.append(line)
    print("Duplicate lines removed")
    
    # remove all lines starting with "Running"
    unique_lines = [line for line in unique_lines if not line.startswith("Running")]
    print("Unwanted lines removed")
    
    # Store only the 2nd word of each line in a list
    beam_nums = [int(line.split()[1]) for line in unique_lines if len(line.split()) > 1]

for i, line in enumerate(unique_lines):
    print(f"{i}: {line}: {beam_nums[i]}")


## Combining filtered catalogues

In [ ]:
directory_main = 'D:\ASKAP Astrometry Storage'
directory_mid = os.path.join(directory_main, 'RACSMid_Queries')
directory_high = os.path.join(directory_main, 'RACSHigh_Queries')

racsmid_rfc_fil_unc = pd.read_csv(f'{directory_mid}/RACSMid1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')
racsmid_rfc_fil = pd.read_csv(f'{directory_mid}/RACSMid1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')

racshigh_rfc_fil_unc = pd.read_csv(f'{directory_high}/RACSHigh1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')
racshigh_rfc_fil = pd.read_csv(f'{directory_high}/RACSHigh1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Thres5.0 arcsec_v5.csv')

In [ ]:
racsmid_rfc_fil = racsmid_rfc_fil.drop_duplicates(subset='JNAME', keep='first')
racshigh_rfc_fil = racshigh_rfc_fil.drop_duplicates(subset='JNAME', keep='first')

In [ ]:
# find the common JNAMEs in racsmid_rfc_fil and racshigh_rfc_fil
common_jnames = set(racsmid_rfc_fil['JNAME']).intersection(set(racshigh_rfc_fil['JNAME']))
print(f'Number of common JNAMEs: {len(common_jnames)}')

In [ ]:
# make new dataframes with only the common JNAMEs
racsmid_rfc_fil_common = racsmid_rfc_fil[racsmid_rfc_fil['JNAME'].isin(common_jnames)]
racshigh_rfc_fil_common = racshigh_rfc_fil[racshigh_rfc_fil['JNAME'].isin(common_jnames)]

# save the dataframes as csv files
racsmid_rfc_fil_common.to_csv(f'{directory_mid}/RACSMid1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Common_v6.csv', index=False)
racshigh_rfc_fil_common.to_csv(f'{directory_high}/RACSHigh1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Common_v6.csv', index=False)

In [ ]:
racsmid_rfc_fil_unc = racsmid_rfc_fil_unc.drop_duplicates(subset='JNAME', keep='first')
racshigh_rfc_fil_unc = racshigh_rfc_fil_unc.drop_duplicates(subset='JNAME', keep='first')

In [ ]:
# find the common JNAMEs in racsmid_rfc_fil_unc and racshigh_rfc_fil_unc
common_jnames_unc = set(racsmid_rfc_fil_unc['JNAME']).intersection(set(racshigh_rfc_fil_unc['JNAME']))
print(f'Number of common JNAMEs in Uncorrected: {len(common_jnames_unc)}')

In [ ]:
# make new dataframes with only the common JNAMEs
racsmid_rfc_fil_common_unc = racsmid_rfc_fil_unc[racsmid_rfc_fil_unc['JNAME'].isin(common_jnames_unc)]
racshigh_rfc_fil_common_unc = racshigh_rfc_fil_unc[racshigh_rfc_fil_unc['JNAME'].isin(common_jnames_unc)]

# save the dataframes as csv files
racsmid_rfc_fil_common_unc.to_csv(f'{directory_mid}/RACSMid1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Common_v6.csv', index=False)
racshigh_rfc_fil_common_unc.to_csv(f'{directory_high}/RACSHigh1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Common_v6.csv', index=False)

## RACSMid1 Uncorrected Filtered and Corrected Filtered Common vs RFC: Per Source

In [ ]:
directory_main = 'D:\ASKAP Astrometry Storage'
directory = os.path.join(directory_main, 'RACSMid_Queries')

racs_rfc_fil_final_unc = pd.read_csv(f'{directory}/RACSMid1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Common_v6.csv')
racs_rfc_fil_final = pd.read_csv(f'{directory}/RACSMid1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Common_v6.csv')

racs_rfc_fil_final_dec30 = racs_rfc_fil_final[racs_rfc_fil_final['DEJ2000'] < 30]

In [ ]:
# Plotting the histograms of the offsets between RACSMid1 uncorrected and RFC catalogues in RA and DEC
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# Plot histogram of RA Offsets (RFC)
axs[0].hist(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'], bins=100, edgecolor='black')
axs[0].set_xlabel('RA Offsets (arcsec)')
axs[0].set_ylabel('Number of Crossmatched Sources')
axs[0].set_title('RACSMid1 Uncorrected vs RFC: RA Offsets')

# Calculate median and confidence intervals
rfc_median_ra = np.nanmedian(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'])
rfc_ci68_ra = np.nanpercentile(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'], [16, 84])
# rfc_ci68_ra = stats.norm.interval(0.68, loc=rfc_median_ra, scale=np.nanstd(racs_rfc_fil_final['RFC RA Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[0].axvline(rfc_median_ra, color='r', linestyle='--', label=f'Median = {rfc_median_ra:.2f} arcsec')
axs[0].axvline(rfc_ci68_ra[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_ra[0]:.2f} to {rfc_ci68_ra[1]:.2f}')
axs[0].axvline(rfc_ci68_ra[1], color='k', linestyle='--')

axs[0].legend(fontsize=8)

# Plot histogram of DEC Offsets (RFC)
axs[1].hist(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'], bins=100, edgecolor='black')
axs[1].set_xlabel('DEC Offsets (arcsec)')
axs[1].set_ylabel('Number of Crossmatched Sources')
axs[1].set_title('RACSMid1 Uncorrected vs RFC: DEC Offsets')

# Calculate median and confidence intervals
rfc_median_dec = np.nanmedian(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'])
rfc_ci68_dec = np.nanpercentile(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'], [16, 84])
# rfc_ci68_dec = stats.norm.interval(0.68, loc=rfc_median_dec, scale=np.nanstd(racs_rfc_fil_final['RFC DEC Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[1].axvline(rfc_median_dec, color='r', linestyle='--', label=f'Median = {rfc_median_dec:.2f} arcsec')
axs[1].axvline(rfc_ci68_dec[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_dec[0]:.2f} to {rfc_ci68_dec[1]:.2f}')
axs[1].axvline(rfc_ci68_dec[1], color='k', linestyle='--')

axs[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Plotting the histograms of the offsets between RACSMid1 corrected and RFC catalogues in RA and DEC
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# Plot histogram of RA Offsets (RFC)
axs[0].hist(racs_rfc_fil_final['RFC RA Offset (arcsec)'], bins=100, edgecolor='black')
axs[0].set_xlabel('RA Offsets (arcsec)')
axs[0].set_ylabel('Number of Crossmatched Sources')
axs[0].set_title('RACSMid1 Corrected vs RFC: RA Offsets')

# Calculate median and confidence intervals
rfc_median_ra = np.nanmedian(racs_rfc_fil_final['RFC RA Offset (arcsec)'])
rfc_ci68_ra = np.nanpercentile(racs_rfc_fil_final['RFC RA Offset (arcsec)'], [16, 84])
# rfc_ci68_ra = stats.norm.interval(0.68, loc=rfc_median_ra, scale=np.nanstd(racs_rfc_fil_final['RFC RA Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[0].axvline(rfc_median_ra, color='r', linestyle='--', label=f'Median = {rfc_median_ra:.2f} arcsec')
axs[0].axvline(rfc_ci68_ra[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_ra[0]:.2f} to {rfc_ci68_ra[1]:.2f}')
axs[0].axvline(rfc_ci68_ra[1], color='k', linestyle='--')

axs[0].legend(fontsize=8)

# Plot histogram of DEC Offsets (RFC)
axs[1].hist(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], bins=100, edgecolor='black')
axs[1].set_xlabel('DEC Offsets (arcsec)')
axs[1].set_ylabel('Number of Crossmatched Sources')
axs[0].set_title('RACSMid1 Corrected vs RFC: DEC Offsets')

# Calculate median and confidence intervals
rfc_median_dec = np.nanmedian(racs_rfc_fil_final['RFC DEC Offset (arcsec)'])
rfc_ci68_dec = np.nanpercentile(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], [16, 84])
# rfc_ci68_dec = stats.norm.interval(0.68, loc=rfc_median_dec, scale=np.nanstd(racs_rfc_fil_final['RFC DEC Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[1].axvline(rfc_median_dec, color='r', linestyle='--', label=f'Median = {rfc_median_dec:.2f} arcsec')
axs[1].axvline(rfc_ci68_dec[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_dec[0]:.2f} to {rfc_ci68_dec[1]:.2f}')
axs[1].axvline(rfc_ci68_dec[1], color='k', linestyle='--')

axs[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Save the offsets and their uncertainties as numpy arrays
np.save(f'{directory}/RACSMid1_Filtered_Common_vs_RFC_Source-wise_Offsets_belowDEC30_Thres{threshold1}_Mean_RA_Offsets.npy', racs_rfc_fil_final_dec30['RFC RA Offset (arcsec)'])
np.save(f'{directory}/RACSMid1_Filtered_Common_vs_RFC_Source-wise_Offsets_belowDEC30_Thres{threshold1}_Mean_DEC_Offsets.npy', racs_rfc_fil_final_dec30['RFC DEC Offset (arcsec)'])

### Comparison of Declination Offsets: Uncorrected vs. Polynomial vs. Final Corrections
This section compares the declination offsets of RACSMid1 against RFC for three cases to evaluate the performance of different correction models.
1. **Uncorrected vs. RFC**: Shows the raw offsets with the 2nd-order polynomial fit (not implemented) overlayed.
2. **Polynomial Corrected vs. RFC**: Shows residuals after applying only the polynomial model.
3. **Final Corrected (User) vs. RFC**: Shows residuals after applying the user's specific corrections.

In [ ]:
from matplotlib import gridspec

# Define the 2nd-order polynomial model from the provided equation
def poly_model(delta):
    """2nd-order polynomial model for declination offsets."""
    return 0 - (0.175 - (0.015 * delta) - (0.0003 * delta**2))

# --- PLACEHOLDERS FOR DATASETS ---
# In the future, these will be loaded from NPY files.
# dec_j2000: Declination of sources (degrees)
# offsets_unc: Uncorrected Declination Offsets (arcsec)
# offsets_user_corr: Final corrected Declination Offsets (arcsec)

dec_j2000_unc = racs_rfc_fil_final_unc['DEJ2000']
offsets_unc = racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)']
dec_j2000 = racs_rfc_fil_final['DEJ2000']
offsets_user_corr = racs_rfc_fil_final['RFC DEC Offset (arcsec)']

# Fit a new 2nd-order polynomial to the uncorrected offsets as a function of declination
# coeffs_fit = np.polyfit(dec_j2000_unc, offsets_unc, 2)
# fit_poly_model = np.poly1d(coeffs_fit)
# print("New fitted polynomial coefficients:", coeffs_fit)  # prints the coefficients for inspection

# Helper function to calculate binned means for the trendline
def get_binned_stats(x, y, bin_width=5):
    """Calculates the mean y-value in declination bins."""
    bins = np.arange(-90, 50 + bin_width, bin_width)
    bin_means, bin_edges, _ = stats.binned_statistic(x, y, statistic='mean', bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    # Filter out bins with no data (NaNs)
    mask = ~np.isnan(bin_means)
    return bin_centers[mask], bin_means[mask]

# Calculate offsets after applying ONLY the polynomial correction to the uncorrected data
# Use the correct declination array for the uncorrected offsets
offsets_poly_only = offsets_unc - poly_model(dec_j2000_unc)

# Calculate offsets after applying ONLY the polynomial correction to the uncorrected data
# Use the correct declination array for the uncorrected offsets
# offsets_poly_new = offsets_unc - fit_poly_model(dec_j2000_unc)

# Dataset configuration for the 3 subplots: (x_data, y_data, title, show_orig_poly, show_fitted_poly, y_label)
plot_configs = [
    (dec_j2000_unc, offsets_unc, "RACSMid1 Uncorrected vs RFC", True, False, "DEC Offsets (arcsec)"),
    (dec_j2000_unc, offsets_poly_only, "RACSMid1 Polynomial Corrected", False, False, "DEC Offsets (arcsec)"),
    # (dec_j2000_unc, offsets_poly_new, "RACSMid1 Fitted Poly Corrected vs RFC", False, False, r"$(\Delta\delta - poly)$ / arcsec"),
    (dec_j2000, offsets_user_corr, "RACSMid1 Corrected vs RFC", False, False, "DEC Offsets (arcsec)")
]

# Create figure with GridSpec for the main scatter plots and collapsed histograms
fig = plt.figure(figsize=(12, 16), dpi=100)
gs = gridspec.GridSpec(4, 2, width_ratios=[4, 1], hspace=0.3, wspace=0.08)

for i, (x_vals, y_vals, title, show_orig_poly, show_fitted_poly, y_label) in enumerate(plot_configs):
    # Left: Offset vs Declination Scatter
    ax_scat = fig.add_subplot(gs[i, 0])
    ax_scat.scatter(x_vals, y_vals, s=4, color='black', alpha=0.3, rasterized=True)

    # Calculate statistics
    mean_val = np.nanmean(y_vals)
    ci68 = np.nanpercentile(y_vals, [16, 84])

    # Reference lines (Mean and 68% CI)
    ax_scat.axhline(mean_val, color='red', linestyle='-', linewidth=1.5, label=f'Mean = {mean_val:.2f}"')
    ax_scat.axhline(ci68[0], color='blue', linestyle='--', linewidth=1, label=f'68% CI = {ci68[0]:.2f}" to {ci68[1]:.2f}"')
    ax_scat.axhline(ci68[1], color='blue', linestyle='--', linewidth=1)

    # polynomial overlays
    delta_range = np.linspace(-90, 50, 200)
    if show_orig_poly:
        ax_scat.plot(delta_range, poly_model(delta_range), color='green', linewidth=4, label='2nd-order Polynomial', alpha=0.8)
    # if show_fitted_poly:
    #     ax_scat.plot(delta_range, fit_poly_model(delta_range), color='orange', linestyle='--', linewidth=4, label='Fitted polynomial', alpha=0.8)

    ax_scat.legend(loc='lower left', fontsize=11, frameon=True, ncol=3)
    ax_scat.set_ylabel(y_label, fontsize=13)
    ax_scat.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax_scat.set_xlim(-92, 52)
    ax_scat.set_ylim(-5, 5)
    ax_scat.tick_params(labelsize=11)
    ax_scat.grid(True, linestyle=':', alpha=0.4)

    if i == 2:
        ax_scat.set_xlabel('DEC (deg)', fontsize=13)

    # Right: Collapsed Histogram
    ax_hist = fig.add_subplot(gs[i, 1], sharey=ax_scat)
    ax_hist.hist(y_vals, bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')
    ax_hist.axhline(mean_val, color='red', linestyle='-', linewidth=1.5)
    ax_hist.axhline(ci68[0], color='blue', linestyle='--', linewidth=1)
    ax_hist.axhline(ci68[1], color='blue', linestyle='--', linewidth=1)

    if i == 2:
        ax_hist.set_xlabel('Count', fontsize=11)
    ax_hist.grid(axis='x', linestyle=':', alpha=0.4)
    plt.setp(ax_hist.get_yticklabels(), visible=False)

plt.tight_layout()
plt.show()

### RFC Offsets: On-Plane vs Off-Plane

In [ ]:
from matplotlib import gridspec

raxx = racs_rfc_fil_final['RAJ2000']
decx = racs_rfc_fil_final['DEJ2000']
coordx_gal = SkyCoord(raxx, decx, unit=u.degree).galactic
b = coordx_gal.b.degree

# Create the 'On-Plane' column directly
racs_rfc_fil_final['On-Plane'] = abs(b) < 10

print(f"Total number of crossmatched sources: {len(racs_rfc_fil_final)}")
print(f"Number of On-Plane Sources: {racs_rfc_fil_final['On-Plane'].sum()}")
print(f"Number of Off-Plane Sources: {len(racs_rfc_fil_final) - racs_rfc_fil_final['On-Plane'].sum()}")

# Create figure with GridSpec for the main scatter plots and collapsed histograms for the on-plane sources and off-plane sources separately
fig = plt.figure(figsize=(12, 16), dpi=100)
gs = gridspec.GridSpec(4, 2, width_ratios=[4, 1], hspace=0.3, wspace=0.08)

ax_scat_on = fig.add_subplot(gs[0, 0])
ax_scat_off = fig.add_subplot(gs[1, 0], sharex=ax_scat_on, sharey=ax_scat_on)

ax_hist_on = fig.add_subplot(gs[0, 1], sharey=ax_scat_on)
ax_hist_off = fig.add_subplot(gs[1, 1], sharey=ax_scat_off)

ax_scat_on.scatter(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['DEJ2000'], racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], s=2, color='black', alpha=0.3, rasterized=True, label='On-Plane Sources')
ax_scat_off.scatter(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['DEJ2000'], racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], s=2, color='black', alpha=0.3, rasterized=True, label='Off-Plane Sources')

median_on = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'])
ci68_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_off = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'])
ci68_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [0.15, 99.85])

# Find RA offsets median and confidence intervals for the on-plane sources and off-plane sources for DEC under 30 deg
median_all_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'])
ci68_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_on_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
ci68_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_off_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
ci68_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [0.15, 99.85])

print(f"Median RA Offset for All Sources under 30 deg DEC: {median_all_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of All Sources under 30 deg DEC: {ci68_all_ra_dec30[0]:.2f}\" to {ci68_all_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of All Sources under 30 deg DEC: {ci_95_all_ra_dec30[0]:.2f}\" to {ci_95_all_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of All Sources under 30 deg DEC: {ci_99_all_ra_dec30[0]:.2f}\" to {ci_99_all_ra_dec30[1]:.2f}\"")
print(f"Median RA Offset for On-Plane Sources under 30 deg DEC: {median_on_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci68_on_ra_dec30[0]:.2f}\" to {ci68_on_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci_95_on_ra_dec30[0]:.2f}\" to {ci_95_on_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci_99_on_ra_dec30[0]:.2f}\" to {ci_99_on_ra_dec30[1]:.2f}\"")
print(f"Median RA Offset for Off-Plane Sources under 30 deg DEC: {median_off_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci68_off_ra_dec30[0]:.2f}\" to {ci68_off_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci_95_off_ra_dec30[0]:.2f}\" to {ci_95_off_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci_99_off_ra_dec30[0]:.2f}\" to {ci_99_off_ra_dec30[1]:.2f}\"")

# RA standard deviation for on-plane sources and off-plane sources for DEC under 30 deg
std_on_ra_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
std_off_ra_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
print(f"Standard Deviation for On-Plane Sources under 30 deg DEC: {std_on_ra_dec30:.2f}\"")
print(f"Standard Deviation for Off-Plane Sources under 30 deg DEC: {std_off_ra_dec30:.2f}\"")

# Find DEC offset median and confidence intervals for the on-plane sources and off-plane sources for DEC under 30 deg
median_all_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'])
ci68_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_on_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
ci68_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_off_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
ci68_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [0.15, 99.85])

print(f"Median DEC Offset for All Sources under 30 deg DEC: {median_all_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of All Sources under 30 deg DEC: {ci68_all_dec30[0]:.2f}\" to {ci68_all_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of All Sources under 30 deg DEC: {ci_95_all_dec30[0]:.2f}\" to {ci_95_all_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of All Sources under 30 deg DEC: {ci_99_all_dec30[0]:.2f}\" to {ci_99_all_dec30[1]:.2f}\"")
print(f"Median DEC Offset for On-Plane Sources under 30 deg DEC: {median_on_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci68_on_dec30[0]:.2f}\" to {ci68_on_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci_95_on_dec30[0]:.2f}\" to {ci_95_on_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci_99_on_dec30[0]:.2f}\" to {ci_99_on_dec30[1]:.2f}\"")
print(f"Median DEC Offset for Off-Plane Sources under 30 deg DEC: {median_off_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci68_off_dec30[0]:.2f}\" to {ci68_off_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci_95_off_dec30[0]:.2f}\" to {ci_95_off_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci_99_off_dec30[0]:.2f}\" to {ci_99_off_dec30[1]:.2f}\"")

# DEC standard deviation for on-plane sources and off-plane sources for DEC under 30 deg
std_on_dec_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
std_off_dec_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
print(f"Standard Deviation for On-Plane Sources under 30 deg DEC: {std_on_dec_dec30:.2f}\"")
print(f"Standard Deviation for Off-Plane Sources under 30 deg DEC: {std_off_dec_dec30:.2f}\"")



print(f"Number of On-Plane Sources within 68% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci68_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci68_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")
print(f"Number of On-Plane Sources within 95% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci_95_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci_95_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")
print(f"Number of On-Plane Sources within 99% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci_99_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci_99_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")

print(f"Number of Off-Plane Sources within 68% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci68_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci68_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")
print(f"Number of Off-Plane Sources within 95% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci_95_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci_95_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")
print(f"Number of Off-Plane Sources within 99% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci_99_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci_99_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")

ax_scat_on.axhline(median_on, color='red', linestyle='-', linewidth=1.5, label=f'On-Plane Median = {median_on:.2f}"')
ax_scat_on.axhline(ci68_on[0], color='red', linestyle='--', linewidth=1, label=f'On-Plane 68% CI = {ci68_on[0]:.2f}" to {ci68_on[1]:.2f}"')
ax_scat_on.axhline(ci68_on[1], color='red', linestyle='--', linewidth=1)
ax_scat_off.axhline(median_off, color='blue', linestyle='-', linewidth=1.5, label=f'Off-Plane Median = {median_off:.2f}"')
ax_scat_off.axhline(ci68_off[0], color='blue', linestyle='--', linewidth=1, label=f'Off-Plane 68% CI = {ci68_off[0]:.2f}" to {ci68_off[1]:.2f}"')
ax_scat_off.axhline(ci68_off[1], color='blue', linestyle='--', linewidth=1)
ax_scat_on.axhline(ci_95_on[0], color='magenta', linestyle=':', linewidth=1, label=f'On-Plane 95% CI = {ci_95_on[0]:.2f}" to {ci_95_on[1]:.2f}"')
ax_scat_on.axhline(ci_95_on[1], color='magenta', linestyle=':', linewidth=1)
ax_scat_off.axhline(ci_95_off[0], color='cyan', linestyle=':', linewidth=1, label=f'Off-Plane 95% CI = {ci_95_off[0]:.2f}" to {ci_95_off[1]:.2f}"')
ax_scat_off.axhline(ci_95_off[1], color='cyan', linestyle=':', linewidth=1)
ax_scat_on.axhline(ci_99_on[0], color='magenta', linestyle='-.', linewidth=1, label=f'On-Plane 99% CI = {ci_99_on[0]:.2f}" to {ci_99_on[1]:.2f}"')
ax_scat_on.axhline(ci_99_on[1], color='magenta', linestyle='-.', linewidth=1)
ax_scat_off.axhline(ci_99_off[0], color='cyan', linestyle='-.', linewidth=1, label=f'Off-Plane 99% CI = {ci_99_off[0]:.2f}" to {ci_99_off[1]:.2f}"') 
ax_scat_off.axhline(ci_99_off[1], color='cyan', linestyle='-.', linewidth=1)

ax_scat_on.legend(loc='upper left', fontsize=11, frameon=True)
ax_scat_off.legend(loc='upper left', fontsize=11, frameon=True)

ax_hist_on.hist(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')
ax_hist_off.hist(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')

ax_hist_on.axhline(median_on, color='red', linestyle='-', linewidth=1.5)
ax_hist_on.axhline(ci68_on[0], color='red', linestyle='--', linewidth=1)
ax_hist_on.axhline(ci68_on[1], color='red', linestyle='--', linewidth=1)
ax_hist_off.axhline(median_off, color='blue', linestyle='-', linewidth=1.5)
ax_hist_off.axhline(ci68_off[0], color='blue', linestyle='--', linewidth=1)
ax_hist_off.axhline(ci68_off[1], color='blue', linestyle='--', linewidth=1)
ax_hist_on.axhline(ci_95_on[0], color='magenta', linestyle=':', linewidth=1)
ax_hist_on.axhline(ci_95_on[1], color='magenta', linestyle=':', linewidth=1)
ax_hist_off.axhline(ci_95_off[0], color='cyan', linestyle=':', linewidth=1)
ax_hist_off.axhline(ci_95_off[1], color='cyan', linestyle=':', linewidth=1)
ax_hist_on.axhline(ci_99_on[0], color='magenta', linestyle='-.', linewidth=1)
ax_hist_on.axhline(ci_99_on[1], color='magenta', linestyle='-.', linewidth=1)
ax_hist_off.axhline(ci_99_off[0], color='cyan', linestyle='-.', linewidth=1)
ax_hist_off.axhline(ci_99_off[1], color='cyan', linestyle='-.', linewidth=1)

plt.show()

In [ ]:
# RA Aitoff plot
ra_deg = racs_rfc_fil_final['RAJ2000']
dec_deg = racs_rfc_fil_final['DEJ2000']
ra_plot = (ra_deg + 180) % 360 - 180  # Convert to -180 to 180 degrees
offset_mag_ra = abs(racs_rfc_fil_final['RFC RA Offset (arcsec)'])

fig1 = plt.figure(figsize=(10, 6))
ax1 = fig1.add_subplot(111, projection='aitoff')
sc1 = ax1.scatter(np.radians(ra_plot), np.radians(dec_deg), c=offset_mag_ra, cmap='viridis', s=1, alpha=0.7)
ax1.set_title('RFC RA Offsets\n')
ax1.grid(True)
cbar1 = plt.colorbar(sc1, ax=ax1, orientation='horizontal', pad=0.05)
cbar1.set_label('RA Offset Magnitude (arcsec)')
plt.show()

# DEC Aitoff plot
ra_deg_dec = racs_rfc_fil_final['RAJ2000']
dec_deg_dec = racs_rfc_fil_final['DEJ2000']
ra_plot_dec = (ra_deg_dec + 180) % 360 - 180
offset_mag_dec = abs(racs_rfc_fil_final['RFC DEC Offset (arcsec)'])

fig2 = plt.figure(figsize=(10, 6))
ax2 = fig2.add_subplot(111, projection='aitoff')
sc2 = ax2.scatter(np.radians(ra_plot_dec), np.radians(dec_deg_dec), c=offset_mag_dec, cmap='viridis', s=1, alpha=0.7)
ax2.set_title('RFC DEC Offsets\n')
ax2.grid(True)
cbar2 = plt.colorbar(sc2, ax=ax2, orientation='horizontal', pad=0.05)
cbar2.set_label('DEC Offset Magnitude (arcsec)')
plt.show()

## RACSHigh1 Uncorrected Filtered and Corrected Filtered Common vs RFC: Per Source

In [ ]:
directory_main = 'D:\ASKAP Astrometry Storage'
directory = os.path.join(directory_main, 'RACSHigh_Queries')

racs_rfc_fil_final_unc = pd.read_csv(f'{directory}/RACSHigh1_Filtered_Uncorrected_vs_RFC_Crossmatched_Sources_Common_v6.csv')
racs_rfc_fil_final = pd.read_csv(f'{directory}/RACSHigh1_Filtered_Corrected_vs_RFC_Crossmatched_Sources_Common_v6.csv')

racs_rfc_fil_final_dec30 = racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]

In [ ]:
# Plotting the histograms of the offsets between RACSHigh1 uncorrected and RFC catalogues in RA and DEC
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# Plot histogram of RA Offsets (RFC)
axs[0].hist(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'], bins=100, edgecolor='black')
axs[0].set_xlabel('RA Offsets (arcsec)')
axs[0].set_ylabel('Number of Crossmatched Sources')
axs[0].set_title('RACSHigh1 Uncorrected vs RFC: RA Offsets')

# Calculate median and confidence intervals
rfc_median_ra = np.nanmedian(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'])
rfc_ci68_ra = np.nanpercentile(racs_rfc_fil_final_unc['Uncorrected RFC RA Offset (arcsec)'], [16, 84])
# rfc_ci68_ra = stats.norm.interval(0.68, loc=rfc_median_ra, scale=np.nanstd(racs_rfc_fil_final['RFC RA Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[0].axvline(rfc_median_ra, color='r', linestyle='--', label=f'Median = {rfc_median_ra:.2f} arcsec')
axs[0].axvline(rfc_ci68_ra[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_ra[0]:.2f} to {rfc_ci68_ra[1]:.2f}')
axs[0].axvline(rfc_ci68_ra[1], color='k', linestyle='--')

axs[0].legend(fontsize=8)

# Plot histogram of DEC Offsets (RFC)
axs[1].hist(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'], bins=100, edgecolor='black')
axs[1].set_xlabel('DEC Offsets (arcsec)')
axs[1].set_ylabel('Number of Crossmatched Sources')
axs[1].set_title('RACSHigh1 Uncorrected vs RFC: DEC Offsets')

# Calculate median and confidence intervals
rfc_median_dec = np.nanmedian(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'])
rfc_ci68_dec = np.nanpercentile(racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)'], [16, 84])
# rfc_ci68_dec = stats.norm.interval(0.68, loc=rfc_median_dec, scale=np.nanstd(racs_rfc_fil_final['RFC DEC Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[1].axvline(rfc_median_dec, color='r', linestyle='--', label=f'Median = {rfc_median_dec:.2f} arcsec')
axs[1].axvline(rfc_ci68_dec[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_dec[0]:.2f} to {rfc_ci68_dec[1]:.2f}')
axs[1].axvline(rfc_ci68_dec[1], color='k', linestyle='--')

axs[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Plotting the histograms of the offsets between RACSHigh1 Corrected and RFC catalogues in RA and DEC
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# Plot histogram of RA Offsets (RFC)
axs[0].hist(racs_rfc_fil_final['RFC RA Offset (arcsec)'], bins=100, edgecolor='black')
axs[0].set_xlabel('RA Offsets (arcsec)')
axs[0].set_ylabel('Number of Crossmatched Sources')
axs[0].set_title('RACSHigh1 Corrected vs RFC: RA Offsets')

# Calculate median and confidence intervals
rfc_median_ra = np.nanmedian(racs_rfc_fil_final['RFC RA Offset (arcsec)'])
rfc_ci68_ra = np.nanpercentile(racs_rfc_fil_final['RFC RA Offset (arcsec)'], [16, 84])
# rfc_ci68_ra = stats.norm.interval(0.68, loc=rfc_median_ra, scale=np.nanstd(racs_rfc_fil_final['RFC RA Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[0].axvline(rfc_median_ra, color='r', linestyle='--', label=f'Median = {rfc_median_ra:.2f} arcsec')
axs[0].axvline(rfc_ci68_ra[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_ra[0]:.2f} to {rfc_ci68_ra[1]:.2f}')
axs[0].axvline(rfc_ci68_ra[1], color='k', linestyle='--')

axs[0].legend(fontsize=8)

# Plot histogram of DEC Offsets (RFC)
axs[1].hist(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], bins=100, edgecolor='black')
axs[1].set_xlabel('DEC Offsets (arcsec)')
axs[1].set_ylabel('Number of Crossmatched Sources')
axs[1].set_title('RACSHigh1 Corrected vs RFC: DEC Offsets')

# Calculate median and confidence intervals
rfc_median_dec = np.nanmedian(racs_rfc_fil_final['RFC DEC Offset (arcsec)'])
rfc_ci68_dec = np.nanpercentile(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], [16, 84])
# rfc_ci68_dec = stats.norm.interval(0.68, loc=rfc_median_dec, scale=np.nanstd(racs_rfc_fil_final['RFC DEC Offset (arcsec)']))

# Draw vertical lines for median and confidence intervals
axs[1].axvline(rfc_median_dec, color='r', linestyle='--', label=f'Median = {rfc_median_dec:.2f} arcsec')
axs[1].axvline(rfc_ci68_dec[0], color='k', linestyle='--', label=f'68% CI = {rfc_ci68_dec[0]:.2f} to {rfc_ci68_dec[1]:.2f}')
axs[1].axvline(rfc_ci68_dec[1], color='k', linestyle='--')

axs[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Save the offsets and their uncertainties as numpy arrays
np.save(f'{directory}/RACSHigh1_Filtered_Common_vs_RFC_Source-wise_Offsets_belowDEC30_Thres{threshold1}_Mean_RA_Offsets.npy', racs_rfc_fil_final_dec30['RFC RA Offset (arcsec)'])
np.save(f'{directory}/RACSHigh1_Filtered_Common_vs_RFC_Source-wise_Offsets_belowDEC30_Thres{threshold1}_Mean_DEC_Offsets.npy', racs_rfc_fil_final_dec30['RFC DEC Offset (arcsec)'])

### Comparison of Declination Offsets: Uncorrected vs. Polynomial vs. Final Corrections
This section compares the declination offsets of RACSHigh1 against RFC for three cases to evaluate the performance of different correction models.
1. **Uncorrected vs. RFC**: Shows the raw offsets with the 4th-order polynomial fit overlayed.
2. **Polynomial Corrected vs. RFC**: Shows residuals after applying only the polynomial model.
3. **Final Corrected (User) vs. RFC**: Shows residuals after applying the user's specific corrections.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

# Define the 4th-order polynomial model from the provided equation
def poly_model(delta):
    """4th-order polynomial model for declination offsets."""
    return 0 - (0.27 - (6.3e-3 * delta) - (5.4e-4 * delta**2) - (8.8e-6 * delta**3) - (5.4e-8 * delta**4))

# --- PLACEHOLDERS FOR DATASETS ---
# In the future, these will be loaded from NPY files.
# dec_j2000: Declination of sources (degrees)
# offsets_unc: Uncorrected Declination Offsets (arcsec)
# offsets_user_corr: Final corrected Declination Offsets (arcsec)

dec_j2000_unc = racs_rfc_fil_final_unc['DEJ2000']
offsets_unc = racs_rfc_fil_final_unc['Uncorrected RFC DEC Offset (arcsec)']
dec_j2000 = racs_rfc_fil_final['DEJ2000']
offsets_user_corr = racs_rfc_fil_final['RFC DEC Offset (arcsec)']

# Fit a new 4th-order polynomial to the uncorrected offsets as a function of declination
# coeffs_fit = np.polyfit(dec_j2000_unc, offsets_unc, 4)
# fit_poly_model = np.poly1d(coeffs_fit)
# print("New fitted polynomial coefficients:", coeffs_fit)  # prints the coefficients for inspection

# Helper function to calculate binned means for the trendline
def get_binned_stats(x, y, bin_width=5):
    """Calculates the mean y-value in declination bins."""
    bins = np.arange(-90, 50 + bin_width, bin_width)
    bin_means, bin_edges, _ = stats.binned_statistic(x, y, statistic='mean', bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    # Filter out bins with no data (NaNs)
    mask = ~np.isnan(bin_means)
    return bin_centers[mask], bin_means[mask]

# Calculate offsets after applying ONLY the polynomial correction to the uncorrected data
# Use the correct declination array for the uncorrected offsets
offsets_poly_only = offsets_unc - poly_model(dec_j2000_unc)

# Calculate offsets after applying ONLY the polynomial correction to the uncorrected data
# Use the correct declination array for the uncorrected offsets
# offsets_poly_new = offsets_unc - fit_poly_model(dec_j2000_unc)

# Dataset configuration for the 3 subplots: (x_data, y_data, title, show_orig_poly, show_fitted_poly, y_label)
plot_configs = [
    (dec_j2000_unc, offsets_unc, "RACSHigh1 Uncorrected vs RFC", True, False, "DEC Offsets (arcsec)"),
    (dec_j2000_unc, offsets_poly_only, "RACSHigh1 Polynomial Corrected", False, False, "DEC Offsets (arcsec)"),
    # (dec_j2000_unc, offsets_poly_new, "RACSHigh1 Fitted Poly Corrected vs RFC", False, False, r"$(\Delta\delta - poly)$ / arcsec"),
    (dec_j2000, offsets_user_corr, "RACSHigh1 Corrected vs RFC", False, False, "DEC Offsets (arcsec)")
]

# Create figure with GridSpec for the main scatter plots and collapsed histograms
fig = plt.figure(figsize=(12, 16), dpi=100)
gs = gridspec.GridSpec(4, 2, width_ratios=[4, 1], hspace=0.3, wspace=0.08)

for i, (x_vals, y_vals, title, show_orig_poly, show_fitted_poly, y_label) in enumerate(plot_configs):
    # Left: Offset vs Declination Scatter
    ax_scat = fig.add_subplot(gs[i, 0])
    ax_scat.scatter(x_vals, y_vals, s=4, color='black', alpha=0.3, rasterized=True)

    # Calculate statistics
    mean_val = np.nanmean(y_vals)
    ci68 = np.nanpercentile(y_vals, [16, 84])

    # Reference lines (Mean and 68% CI)
    ax_scat.axhline(mean_val, color='red', linestyle='-', linewidth=1.5, label=f'Mean = {mean_val:.2f}"')
    ax_scat.axhline(ci68[0], color='blue', linestyle='--', linewidth=1, label=f'68% CI = {ci68[0]:.2f}" to {ci68[1]:.2f}"')
    ax_scat.axhline(ci68[1], color='blue', linestyle='--', linewidth=1)

    # polynomial overlays
    delta_range = np.linspace(-90, 50, 200)
    if show_orig_poly:
        ax_scat.plot(delta_range, poly_model(delta_range), color='green', linewidth=4, label='4th-order Polynomial', alpha=0.8)
    # if show_fitted_poly:
    #     ax_scat.plot(delta_range, fit_poly_model(delta_range), color='orange', linestyle='--', linewidth=4, label='Fitted polynomial', alpha=0.8)

    ax_scat.legend(loc='lower left', fontsize=11, frameon=True, ncol=3)
    ax_scat.set_ylabel(y_label, fontsize=13)
    ax_scat.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax_scat.set_xlim(-92, 52)
    ax_scat.set_ylim(-5, 5)
    ax_scat.tick_params(labelsize=11)
    ax_scat.grid(True, linestyle=':', alpha=0.4)

    if i == 2:
        ax_scat.set_xlabel('DEC (deg)', fontsize=13)

    # Right: Collapsed Histogram
    ax_hist = fig.add_subplot(gs[i, 1], sharey=ax_scat)
    ax_hist.hist(y_vals, bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')
    ax_hist.axhline(mean_val, color='red', linestyle='-', linewidth=1.5)
    ax_hist.axhline(ci68[0], color='blue', linestyle='--', linewidth=1)
    ax_hist.axhline(ci68[1], color='blue', linestyle='--', linewidth=1)

    if i == 2:
        ax_hist.set_xlabel('Count', fontsize=11)
    ax_hist.grid(axis='x', linestyle=':', alpha=0.4)
    plt.setp(ax_hist.get_yticklabels(), visible=False)

plt.tight_layout()
plt.show()

### RFC Offsets: On-Plane vs Off-Plane

In [ ]:
from matplotlib import gridspec

raxx = racs_rfc_fil_final['RAJ2000']
decx = racs_rfc_fil_final['DEJ2000']
coordx_gal = SkyCoord(raxx, decx, unit=u.degree).galactic
b = coordx_gal.b.degree

# Create the 'On-Plane' column directly
racs_rfc_fil_final['On-Plane'] = abs(b) < 10

print(f"Total number of crossmatched sources: {len(racs_rfc_fil_final)}")
print(f"Number of On-Plane Sources: {racs_rfc_fil_final['On-Plane'].sum()}")
print(f"Number of Off-Plane Sources: {len(racs_rfc_fil_final) - racs_rfc_fil_final['On-Plane'].sum()}")

# Create figure with GridSpec for the main scatter plots and collapsed histograms for the on-plane sources and off-plane sources separately
fig = plt.figure(figsize=(12, 16), dpi=100)
gs = gridspec.GridSpec(4, 2, width_ratios=[4, 1], hspace=0.3, wspace=0.08)

ax_scat_on = fig.add_subplot(gs[0, 0])
ax_scat_off = fig.add_subplot(gs[1, 0], sharex=ax_scat_on, sharey=ax_scat_on)

ax_hist_on = fig.add_subplot(gs[0, 1], sharey=ax_scat_on)
ax_hist_off = fig.add_subplot(gs[1, 1], sharey=ax_scat_off)

ax_scat_on.scatter(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['DEJ2000'], racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], s=2, color='black', alpha=0.3, rasterized=True, label='On-Plane Sources')
ax_scat_off.scatter(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['DEJ2000'], racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], s=2, color='black', alpha=0.3, rasterized=True, label='Off-Plane Sources')

median_on = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'])
ci68_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_on = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_off = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'])
ci68_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_off = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [0.15, 99.85])

# Find RA offsets median and confidence intervals for the on-plane sources and off-plane sources all-sky
median_all_ra = np.nanmedian(racs_rfc_fil_final['RFC RA Offset (arcsec)'])
ci68_all_ra = np.nanpercentile(racs_rfc_fil_final['RFC RA Offset (arcsec)'], [16, 84])
ci_95_all_ra = np.nanpercentile(racs_rfc_fil_final['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_all_ra = np.nanpercentile(racs_rfc_fil_final['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_on_ra = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC RA Offset (arcsec)'])
ci68_on_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_on_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_on_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_off_ra = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC RA Offset (arcsec)'])
ci68_off_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_off_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_off_ra = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC RA Offset (arcsec)'], [0.15, 99.85])

print(f"Median RA Offset for All Sources: {median_all_ra:.2f}\"")
print(f"68% CI for RA Offsets of All Sources: {ci68_all_ra[0]:.2f}\" to {ci68_all_ra[1]:.2f}\"")
print(f"95% CI for RA Offsets of All Sources: {ci_95_all_ra[0]:.2f}\" to {ci_95_all_ra[1]:.2f}\"")
print(f"99% CI for RA Offsets of All Sources: {ci_99_all_ra[0]:.2f}\" to {ci_99_all_ra[1]:.2f}\"")
print(f"Median RA Offset for On-Plane Sources: {median_on_ra:.2f}\"")
print(f"68% CI for RA Offsets of On-Plane Sources: {ci68_on_ra[0]:.2f}\" to {ci68_on_ra[1]:.2f}\"")
print(f"95% CI for RA Offsets of On-Plane Sources: {ci_95_on_ra[0]:.2f}\" to {ci_95_on_ra[1]:.2f}\"")
print(f"99% CI for RA Offsets of On-Plane Sources: {ci_99_on_ra[0]:.2f}\" to {ci_99_on_ra[1]:.2f}\"")
print(f"Median RA Offset for Off-Plane Sources: {median_off_ra:.2f}\"")
print(f"68% CI for RA Offsets of Off-Plane Sources: {ci68_off_ra[0]:.2f}\" to {ci68_off_ra[1]:.2f}\"")
print(f"95% CI for RA Offsets of Off-Plane Sources: {ci_95_off_ra[0]:.2f}\" to {ci_95_off_ra[1]:.2f}\"")
print(f"99% CI for RA Offsets of Off-Plane Sources: {ci_99_off_ra[0]:.2f}\" to {ci_99_off_ra[1]:.2f}\"")

# Find RA offsets median and confidence intervals for the on-plane sources and off-plane sources for DEC under 30 deg
median_all_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'])
ci68_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_all_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_on_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
ci68_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_on_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [0.15, 99.85])
median_off_ra_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
ci68_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [16, 84])
ci_95_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [2.3, 97.7])
ci_99_off_ra_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'], [0.15, 99.85])

print(f"Median RA Offset for All Sources under 30 deg DEC: {median_all_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of All Sources under 30 deg DEC: {ci68_all_ra_dec30[0]:.2f}\" to {ci68_all_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of All Sources under 30 deg DEC: {ci_95_all_ra_dec30[0]:.2f}\" to {ci_95_all_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of All Sources under 30 deg DEC: {ci_99_all_ra_dec30[0]:.2f}\" to {ci_99_all_ra_dec30[1]:.2f}\"")
print(f"Median RA Offset for On-Plane Sources under 30 deg DEC: {median_on_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci68_on_ra_dec30[0]:.2f}\" to {ci68_on_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci_95_on_ra_dec30[0]:.2f}\" to {ci_95_on_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of On-Plane Sources under 30 deg DEC: {ci_99_on_ra_dec30[0]:.2f}\" to {ci_99_on_ra_dec30[1]:.2f}\"")
print(f"Median RA Offset for Off-Plane Sources under 30 deg DEC: {median_off_ra_dec30:.2f}\"")
print(f"68% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci68_off_ra_dec30[0]:.2f}\" to {ci68_off_ra_dec30[1]:.2f}\"")
print(f"95% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci_95_off_ra_dec30[0]:.2f}\" to {ci_95_off_ra_dec30[1]:.2f}\"")
print(f"99% CI for RA Offsets of Off-Plane Sources under 30 deg DEC: {ci_99_off_ra_dec30[0]:.2f}\" to {ci_99_off_ra_dec30[1]:.2f}\"")

# RA standard deviation for on-plane sources and off-plane sources for DEC under 30 deg
std_on_ra_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
std_off_ra_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC RA Offset (arcsec)'])
print(f"Standard Deviation for On-Plane Sources under 30 deg DEC: {std_on_ra_dec30:.2f}\"")
print(f"Standard Deviation for Off-Plane Sources under 30 deg DEC: {std_off_ra_dec30:.2f}\"")


# Find DEC offset median and confidence intervals for the on-plane sources and off-plane sources all-sky
median_all_dec = np.nanmedian(racs_rfc_fil_final['RFC DEC Offset (arcsec)'])
ci68_all_dec = np.nanpercentile(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_all_dec = np.nanpercentile(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_all_dec = np.nanpercentile(racs_rfc_fil_final['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_on_dec = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'])
ci68_on_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_on_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_on_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_off_dec = np.nanmedian(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'])
ci68_off_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_off_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_off_dec = np.nanpercentile(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], [0.15, 99.85])

print(f"Median DEC Offset for All Sources: {median_all_dec:.2f}\"")
print(f"68% CI for DEC Offsets of All Sources: {ci68_all_dec[0]:.2f}\" to {ci68_all_dec[1]:.2f}\"")
print(f"95% CI for DEC Offsets of All Sources: {ci_95_all_dec[0]:.2f}\" to {ci_95_all_dec[1]:.2f}\"")
print(f"99% CI for DEC Offsets of All Sources: {ci_99_all_dec[0]:.2f}\" to {ci_99_all_dec[1]:.2f}\"")
print(f"Median DEC Offset for On-Plane Sources: {median_on_dec:.2f}\"")
print(f"68% CI for DEC Offsets of On-Plane Sources: {ci68_on_dec[0]:.2f}\" to {ci68_on_dec[1]:.2f}\"")
print(f"95% CI for DEC Offsets of On-Plane Sources: {ci_95_on_dec[0]:.2f}\" to {ci_95_on_dec[1]:.2f}\"")
print(f"99% CI for DEC Offsets of On-Plane Sources: {ci_99_on_dec[0]:.2f}\" to {ci_99_on_dec[1]:.2f}\"")
print(f"Median DEC Offset for Off-Plane Sources: {median_off_dec:.2f}\"")
print(f"68% CI for DEC Offsets of Off-Plane Sources: {ci68_off_dec[0]:.2f}\" to {ci68_off_dec[1]:.2f}\"")
print(f"95% CI for DEC Offsets of Off-Plane Sources: {ci_95_off_dec[0]:.2f}\" to {ci_95_off_dec[1]:.2f}\"")
print(f"99% CI for DEC Offsets of Off-Plane Sources: {ci_99_off_dec[0]:.2f}\" to {ci_99_off_dec[1]:.2f}\"")

# Find DEC offset median and confidence intervals for the on-plane sources and off-plane sources for DEC under 30 deg
median_all_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'])
ci68_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_all_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['DEJ2000']) < 30]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_on_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
ci68_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_on_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [0.15, 99.85])
median_off_dec30 = np.nanmedian(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
ci68_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [16, 84])
ci_95_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [2.3, 97.7])
ci_99_off_dec30 = np.nanpercentile(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'], [0.15, 99.85])

print(f"Median DEC Offset for All Sources under 30 deg DEC: {median_all_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of All Sources under 30 deg DEC: {ci68_all_dec30[0]:.2f}\" to {ci68_all_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of All Sources under 30 deg DEC: {ci_95_all_dec30[0]:.2f}\" to {ci_95_all_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of All Sources under 30 deg DEC: {ci_99_all_dec30[0]:.2f}\" to {ci_99_all_dec30[1]:.2f}\"")
print(f"Median DEC Offset for On-Plane Sources under 30 deg DEC: {median_on_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci68_on_dec30[0]:.2f}\" to {ci68_on_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci_95_on_dec30[0]:.2f}\" to {ci_95_on_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of On-Plane Sources under 30 deg DEC: {ci_99_on_dec30[0]:.2f}\" to {ci_99_on_dec30[1]:.2f}\"")
print(f"Median DEC Offset for Off-Plane Sources under 30 deg DEC: {median_off_dec30:.2f}\"")
print(f"68% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci68_off_dec30[0]:.2f}\" to {ci68_off_dec30[1]:.2f}\"")
print(f"95% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci_95_off_dec30[0]:.2f}\" to {ci_95_off_dec30[1]:.2f}\"")
print(f"99% CI for DEC Offsets of Off-Plane Sources under 30 deg DEC: {ci_99_off_dec30[0]:.2f}\" to {ci_99_off_dec30[1]:.2f}\"")



# DEC standard deviation for on-plane sources and off-plane sources for DEC under 30 deg
std_on_dec_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == True) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
std_off_dec_dec30 = np.nanstd(racs_rfc_fil_final[(racs_rfc_fil_final['On-Plane'] == False) & ((racs_rfc_fil_final['DEJ2000']) < 30)]['RFC DEC Offset (arcsec)'])
print(f"Standard Deviation for On-Plane Sources under 30 deg DEC: {std_on_dec_dec30:.2f}\"")
print(f"Standard Deviation for Off-Plane Sources under 30 deg DEC: {std_off_dec_dec30:.2f}\"")

print(f"Number of On-Plane Sources within 68% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci68_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci68_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")
print(f"Number of On-Plane Sources within 95% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci_95_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci_95_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")
print(f"Number of On-Plane Sources within 99% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] >= ci_99_on[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'] <= ci_99_on[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True].shape[0]}")

print(f"Number of Off-Plane Sources within 68% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci68_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci68_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")
print(f"Number of Off-Plane Sources within 95% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci_95_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci_95_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")
print(f"Number of Off-Plane Sources within 99% CI: {np.sum((racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] >= ci_99_off[0]) & (racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'] <= ci_99_off[1]))} out of {racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False].shape[0]}")

ax_scat_on.axhline(median_on, color='red', linestyle='-', linewidth=1.5, label=f'On-Plane Median = {median_on:.2f}"')
ax_scat_on.axhline(ci68_on[0], color='red', linestyle='--', linewidth=1, label=f'On-Plane 68% CI = {ci68_on[0]:.2f}" to {ci68_on[1]:.2f}"')
ax_scat_on.axhline(ci68_on[1], color='red', linestyle='--', linewidth=1)
ax_scat_off.axhline(median_off, color='blue', linestyle='-', linewidth=1.5, label=f'Off-Plane Median = {median_off:.2f}"')
ax_scat_off.axhline(ci68_off[0], color='blue', linestyle='--', linewidth=1, label=f'Off-Plane 68% CI = {ci68_off[0]:.2f}" to {ci68_off[1]:.2f}"')
ax_scat_off.axhline(ci68_off[1], color='blue', linestyle='--', linewidth=1)
ax_scat_on.axhline(ci_95_on[0], color='magenta', linestyle=':', linewidth=1, label=f'On-Plane 95% CI = {ci_95_on[0]:.2f}" to {ci_95_on[1]:.2f}"')
ax_scat_on.axhline(ci_95_on[1], color='magenta', linestyle=':', linewidth=1)
ax_scat_off.axhline(ci_95_off[0], color='cyan', linestyle=':', linewidth=1, label=f'Off-Plane 95% CI = {ci_95_off[0]:.2f}" to {ci_95_off[1]:.2f}"')
ax_scat_off.axhline(ci_95_off[1], color='cyan', linestyle=':', linewidth=1)
ax_scat_on.axhline(ci_99_on[0], color='magenta', linestyle='-.', linewidth=1, label=f'On-Plane 99% CI = {ci_99_on[0]:.2f}" to {ci_99_on[1]:.2f}"')
ax_scat_on.axhline(ci_99_on[1], color='magenta', linestyle='-.', linewidth=1)
ax_scat_off.axhline(ci_99_off[0], color='cyan', linestyle='-.', linewidth=1, label=f'Off-Plane 99% CI = {ci_99_off[0]:.2f}" to {ci_99_off[1]:.2f}"') 
ax_scat_off.axhline(ci_99_off[1], color='cyan', linestyle='-.', linewidth=1)

ax_scat_on.legend(loc='upper left', fontsize=11, frameon=True)
ax_scat_off.legend(loc='upper left', fontsize=11, frameon=True)

ax_hist_on.hist(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == True]['RFC DEC Offset (arcsec)'], bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')
ax_hist_off.hist(racs_rfc_fil_final[racs_rfc_fil_final['On-Plane'] == False]['RFC DEC Offset (arcsec)'], bins=100, orientation='horizontal', color='black', alpha=0.8, histtype='stepfilled')

ax_hist_on.axhline(median_on, color='red', linestyle='-', linewidth=1.5)
ax_hist_on.axhline(ci68_on[0], color='red', linestyle='--', linewidth=1)
ax_hist_on.axhline(ci68_on[1], color='red', linestyle='--', linewidth=1)
ax_hist_off.axhline(median_off, color='blue', linestyle='-', linewidth=1.5)
ax_hist_off.axhline(ci68_off[0], color='blue', linestyle='--', linewidth=1)
ax_hist_off.axhline(ci68_off[1], color='blue', linestyle='--', linewidth=1)
ax_hist_on.axhline(ci_95_on[0], color='magenta', linestyle=':', linewidth=1)
ax_hist_on.axhline(ci_95_on[1], color='magenta', linestyle=':', linewidth=1)
ax_hist_off.axhline(ci_95_off[0], color='cyan', linestyle=':', linewidth=1)
ax_hist_off.axhline(ci_95_off[1], color='cyan', linestyle=':', linewidth=1)
ax_hist_on.axhline(ci_99_on[0], color='magenta', linestyle='-.', linewidth=1)
ax_hist_on.axhline(ci_99_on[1], color='magenta', linestyle='-.', linewidth=1)
ax_hist_off.axhline(ci_99_off[0], color='cyan', linestyle='-.', linewidth=1)
ax_hist_off.axhline(ci_99_off[1], color='cyan', linestyle='-.', linewidth=1)

plt.show()

In [ ]:
# RA Aitoff plot
ra_deg = racs_rfc_fil_final['RAJ2000']
dec_deg = racs_rfc_fil_final['DEJ2000']
ra_plot = (ra_deg + 180) % 360 - 180  # Convert to -180 to 180 degrees
offset_mag_ra = abs(racs_rfc_fil_final['RFC RA Offset (arcsec)'])

fig1 = plt.figure(figsize=(10, 6))
ax1 = fig1.add_subplot(111, projection='aitoff')
sc1 = ax1.scatter(np.radians(ra_plot), np.radians(dec_deg), c=offset_mag_ra, cmap='viridis', s=1, alpha=0.7)
ax1.set_title('RFC RA Offsets\n')
ax1.grid(True)
cbar1 = plt.colorbar(sc1, ax=ax1, orientation='horizontal', pad=0.05)
cbar1.set_label('RA Offset Magnitude (arcsec)')
plt.show()

# DEC Aitoff plot
ra_deg_dec = racs_rfc_fil_final['RAJ2000']
dec_deg_dec = racs_rfc_fil_final['DEJ2000']
ra_plot_dec = (ra_deg_dec + 180) % 360 - 180
offset_mag_dec = abs(racs_rfc_fil_final['RFC DEC Offset (arcsec)'])

fig2 = plt.figure(figsize=(10, 6))
ax2 = fig2.add_subplot(111, projection='aitoff')
sc2 = ax2.scatter(np.radians(ra_plot_dec), np.radians(dec_deg_dec), c=offset_mag_dec, cmap='viridis', s=1, alpha=0.7)
ax2.set_title('RFC DEC Offsets\n')
ax2.grid(True)
cbar2 = plt.colorbar(sc2, ax=ax2, orientation='horizontal', pad=0.05)
cbar2.set_label('DEC Offset Magnitude (arcsec)')
plt.show()